🟢 < Module 5.1 > NumPy & Pandas — High-Performance Data Ingestion, Merging & Reshaping

#### 🎯 Context & Objectives

In real-world data science pipelines, analytical data arrives fragmented across multiple sources (macroeconomic files from Eurostat, individual survey microdata waves from the European Social Survey, and scraped categorical attributes). To feed downstream machine learning models without introducing memory leaks or data corruption, a Data Scientist must master vectorized data manipulation, memory-efficient type casting, and relational merging using NumPy and Pandas.

In this exercise, you will build an optimized ingestion pipeline that merges multi-source datasets, restructures data shapes using pivot/melt operations, and handles complex multi-index groupings.

#### 📑 Exercise Instructions

##### 1. Syntactic / Algorithmic Level (Vectorized Operations & Reshaping)

Create a Python notebook or script (`notebooks/05_1_numpy_pandas_operations.ipynb`) that implements the following technical steps:

- **Vectorized Array Operations**:
    
    - Perform high-performance vectorized calculations on NumPy arrays (e.g., calculating standardized z-scores across multi-dimensional macro matrices without explicit Python `for` loops).
        
- **Data Reshaping (Wide vs. Long Formats)**:
    
    - Take a wide Eurostat macroeconomic table (where years are represented as individual columns) and transform it into a clean long-format tidy dataset using `pd.melt()`.
        
    - Reconstruct the wide matrix back using `pd.pivot_table()`, ensuring multi-level column indexing is properly flattened and cleaned.
        

##### 2. Applied / Data Level (Multi-Source Relational Merging)

Apply advanced Pandas joining techniques to your project datasets:

- **Relational Merging & Key Alignment**:
    
    - Load your cleaned CSV/DuckDB exports for country metadata (`GlobalCountriesLang.csv`), macroeconomic features, and ESS survey summaries.
        
    - Perform inner and outer joins (`pd.merge()`) using compound keys (e.g., country code `cntry` / `geo` and survey year/round) to prevent row explosion or silent data loss.
        
- **Hierarchical Groupings & Aggregations**:
    
    - Group the merged master dataset by macro-economic region clusters (`cntgrp_fc`) and compute multi-metric aggregated summaries (mean, median, IQR) for institutional trust variables.
        

#### 🌐 Official References for Documentation

- **Pandas Merging / Joining Guide**: [Pandas Merge, Join, and Concatenate Documentation](https://pandas.pydata.org/docs/user_guide/merging.html)
    
- **Pandas Reshaping Guide**: [Reshaping and Pivot Tables Documentation](https://pandas.pydata.org/docs/user_guide/reshaping.html)
    

#### 📚 Recommended Manuals & CheatSheets

- **Pandas Cheat Sheet**: [DataCamp Pandas Cheat Sheet (PDF)](https://www.google.com/search?q=https://www.datacamp.com/cheat-sheet/pandas-cheat-sheet)
    

💻 **Your Challenge:** Crea el cuaderno `notebooks/05_1_numpy_pandas_operations.ipynb` en tu entorno local, ejecuta las transformaciones y fusiones de alto rendimiento con tus datasets, y pásame tu código o resultados cuando estés listo para avanzar al Módulo 6 (Data Cleaning & Imputation).

In [1]:
# I will focus this excersise to practice this vectorized methods, but aswell, to link indirectly , different dataframes using variables that will create different classes. We cant invent or
# simulate clases, we will use some variables related between dataframes to make indirectly different clases with a dense meaning. For example age, number of sons, how many
# people lives in his home, etc. For this i upload the whole csv to my database via python on a code cell or a bash terminal from activated venv with required libraries.
import duckdb as db
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()  # search .env on the directory
con = db.connect()
con.execute("ATTACH '' AS db (TYPE postgres, SCHEMA 'public')")

In [2]:
#Let's inspect how is the format of our csv's located in the sql databaseÑ
schema_info = con.sql("DESCRIBE db.PobrezaEU").df()
print(schema_info)

                               column_name column_type null   key default  \
0                                STRUCTURE     VARCHAR  YES  None    None   
1                             STRUCTURE_ID     VARCHAR  YES  None    None   
2                           STRUCTURE_NAME     VARCHAR  YES  None    None   
3                                     freq     VARCHAR  YES  None    None   
4                           Time frequency     VARCHAR  YES  None    None   
5                                     unit     VARCHAR  YES  None    None   
6                          Unit of measure     VARCHAR  YES  None    None   
7                                      age     VARCHAR  YES  None    None   
8                                Age class     VARCHAR  YES  None    None   
9                                      sex     VARCHAR  YES  None    None   
10                                   Sex_1     VARCHAR  YES  None    None   
11                                     geo     VARCHAR  YES  None    None   

In [5]:
import numpy as np
#1 lets get z-score from PobrezaEu.csv located in our local database.
# A. Lets define the query sentence.
query = """
    SELECT TIME_PERIOD, OBS_VALUE, geo 
    FROM db.pobrezaeu
    WHERE OBS_VALUE IS NOT NULL AND TIME_PERIOD >= 2015
"""
df_filtered = con.sql(query).df()

# B. Direct conversion to Numpy arrais for the vectorized computing.
time_arr = df_filtered['TIME_PERIOD'].to_numpy(dtype=int)
obs_arr = df_filtered['OBS_VALUE'].to_numpy(dtype=float)
geo_arr = df_filtered['geo'].to_numpy(dtype=str)

# C. Vectorized calculation using Numpy Z-scores
mean_obs = np.mean(obs_arr)
std_obs = np.std(obs_arr)

z_scores = (obs_arr - mean_obs) / std_obs

print(f"✅ Succesfull arrays exctracted via DuckDB:")
print(f" - Computed Samples: {len(obs_arr)}")
print(f" - Array mean: {mean_obs:.4f}")
print(f" - First 5 z-score results: {z_scores[:5]}")

✅ Succesfull arrays exctracted via DuckDB:
 - Computed Samples: 87720
 - Array mean: 1419.7538
 - First 5 z-score results: [-0.21022835 -0.2109397  -0.21138817 -0.21211498 -0.2120222 ]


In [8]:
#2. Applied / Data Level (Multi-Source Relational Merging) using unique key values from common columms of our different datablocks.
# We will use our Surveys DF and our matrix economic and regional features we did exercises ago.
# We use a query on sql via duckdb to make an easy filtering:
import duckdb

# Asegúrate de tener tu conexión 'con' vinculada a PostgreSQL
query = """
    SELECT 
        e.cntry,
        e.essround,
        e.trstlgl,
        e.lrscale,
        m.TIME_PERIOD,
        m.OBS_VALUE as macro_indicator
    FROM db.surveys e
    LEFT JOIN db.macro_features_matrix_2015_2025 m
        ON e.cntry = m.geo
    WHERE e.cntry IS NOT NULL
"""

# Ejecutamos la consulta y la traemos a un DataFrame de Pandas o la convertimos a NumPy
df_merged = con.sql(query).df()
print(f"✅ ¡Merge completado con éxito! Dimensiones resultantes: {df_merged.shape}")
print(df_merged.head())


BinderException: Binder Error: Table "m" does not have a column named "TIME_PERIOD"

Candidate bindings: : "geo"